# AGender-MTL - Step 2: oversampling the rare age groups (CPU)

Does adding synthetic training rows help the network and Gradient Boosting find the rare age groups?

- **Same start as Step 1:** the same split, encoding and QUBO choice for every seed, made once before any
  oversampling, so every method uses the same rows and the same attributes. Only training rows are
  oversampled; validation and test rows stay real.
- **Methods:** none (class weights only, as in Step 1); SMOTE with the earlier setting (the two smallest
  age groups raised to the median size); random oversampling; SMOTE-NC; SMOTE-NC on the age label only;
  SMOTE-NC with full balance; Borderline-SMOTE; ADASYN; CTGAN.
- **Label:** except where named, the joint label gender x age group is balanced, so the gender balance
  does not move. Unless named, classes below the median size are raised to the median size.
- **Age schemes:** 6 and 10 classes; for 48 classes only the earlier SMOTE setting. **Seeds:** 42, 7, 123, 2024, 99.
- **Scores (test rows):** macro-F1, balanced accuracy, recall in the two smallest age groups, recall in
  every group, +-1 accuracy, AUC, ECE; gender AUC and ECE; each method against "none" over the same seeds.
- **Checks of the synthetic rows:** distance to the closest real row, and how well a classifier tells
  synthetic rows from real ones.

**Kaggle settings:** Accelerator = **None** (CPU). **Internet = On** (only for installing `ctgan`; without
it the CTGAN rows are skipped and everything else runs).
**Inputs to add:** `agender-mtl-code`, and the Step 0 output: either the dataset `agender-mtl-step0`, or
the notebook itself (Add Input -> Notebooks -> step0_data).
**Expected run time:** about 2-3 hours. Every (scheme, method) is written as soon as it is done.

In [ ]:
import os
import sys
from pathlib import Path

# The code is attached as the Kaggle dataset `agender-mtl-code`. Find the `agender_mtl` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('benchmark.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-mtl-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-mtl-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender_mtl' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender_mtl`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender_mtl').exists():
        os.symlink(package.parent, code_root / 'agender_mtl')

sys.path.insert(0, str(code_root))
import agender_mtl
print('Code loaded from', Path(agender_mtl.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_mtl_outputs/step2')
TIME_BUDGET_MINUTES = 330
SCHEMES = ['6', '10']
METHODS = ['none', 'SMOTE, earlier setting', 'random oversampling', 'SMOTE-NC', 'SMOTE-NC, age label only',
           'SMOTE-NC, full balance', 'Borderline-SMOTE', 'ADASYN', 'CTGAN']
CTGAN_EPOCHS = 100
# 48 classes: only the earlier SMOTE setting, to re-produce the row "MTL + SMOTE" of the earlier 48-class table
EXTRA = {'48': ['none', 'SMOTE, earlier setting']}

In [ ]:
import importlib.util
import subprocess

# imbalanced-learn is part of the Kaggle image; ctgan is not, so install it when the internet is on.
for package, module in (('imbalanced-learn', 'imblearn'), ('ctgan', 'ctgan')):
    if importlib.util.find_spec(module) is None:
        done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], capture_output=True, text=True)
        print(package, 'installed' if done.returncode == 0 else 'could not be installed: ' + done.stderr[-500:])
if importlib.util.find_spec('ctgan') is None:
    METHODS = [m for m in METHODS if m != 'CTGAN']
    print('CTGAN skipped (no internet). Methods:', METHODS)

In [ ]:
import json

from agender_mtl import environment, step1, step2
from agender_mtl.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'methods': METHODS, 'schemes': SCHEMES}

# Kaggle unpacks zip files when an output is saved as a dataset, so look for the zip or the csv inside it.
lat_path = find_file('lat_s1.zip', INPUT) or find_file('age-gender-pred-dataset.csv', INPUT)
print('LAT file:', lat_path)
if lat_path is None:
    raise RuntimeError('Attach the output of Step 0 (dataset agender-mtl-step0, or the step0_data notebook): it holds the LAT file.')

summary['environment'] = runner.run('measure the machine', environment.collect)
raw = runner.run('read LAT', step1.raw_lat, lat_path)
prepared = step2.Prepared(raw)
print(raw.shape)

In [ ]:
# The 48-class pass comes before CTGAN, and CTGAN comes last: it is the slowest, so a time limit cuts it first.
plan = [(scheme, m) for m in METHODS if m != 'CTGAN' for scheme in SCHEMES]
plan += [(scheme, m) for scheme, methods in EXTRA.items() for m in methods]
plan += [(scheme, m) for m in METHODS if m == 'CTGAN' for scheme in SCHEMES]
for scheme, method in plan:
    result = runner.run(f'{scheme}-class scheme / {method}', step2.run_method, prepared, scheme, method, OUTPUT,
                        ctgan_epochs=CTGAN_EPOCHS)
    summary[f'{scheme}class / {method}'] = result
    if result:
        print('  rows added per seed:', result['rows_added'])
        for model, scores in result['test_age_means'].items():
            print(f'  {model:18s}', scores)

In [ ]:
# ---- Tables and summary (always written) ----
runner.run('QUBO choices', step2.chosen_attributes, prepared, OUTPUT)
summary['combined'] = runner.run('combine the parts', step2.combine, OUTPUT)
summary['steps'] = runner.steps
with open(OUTPUT / 'step2_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')